<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week6_online_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 online session – multiple regression case studies

Companion notebook to the Week 6 online recording. Run each cell as you watch: the numbers on the slides are computed here.

Data: the textbook's CPS earnings data (Békés & Kézdi Ch. 10, 18,241 people with a graduate degree), then the Vienna hotels from Weeks 4 and 5.

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import gaussian_kde

base = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/"
cps_df = pd.read_csv(base + "cps_gender_pay_gap.csv")
print(len(cps_df), "people")
cps_df.head()

# Part 1: the gender pay gap

## Adding age as a confounder

The gap alone, the gap holding age fixed, and age regressed on female.

In [ ]:
m1 = smf.ols('ln_wage ~ female', data=cps_df).fit(cov_type='HC1')
m2 = smf.ols('ln_wage ~ female + age', data=cps_df).fit(cov_type='HC1')
aux = smf.ols('age ~ female', data=cps_df).fit(cov_type='HC1')
print(m1.params.round(3), m2.params.round(3), aux.params.round(3), sep='\n\n')

## Checking the OVB formula

β − β₁ should equal δ × β₂.

In [ ]:
print(m1.params['female'] - m2.params['female'])
print(aux.params['female'] * m2.params['age'])

## The age distribution by gender

In [ ]:
age_grid = np.linspace(cps_df['age'].min(), cps_df['age'].max(), 200)
fig, ax = plt.subplots(figsize=(7, 3))
for f, colour, label in [(0, '#2d6a4f', 'Male'), (1, '#457b9d', 'Female')]:
    kde = gaussian_kde(cps_df.loc[cps_df['female'] == f, 'age'])
    ax.plot(age_grid, kde(age_grid), color=colour, linewidth=2.5, label=label)
ax.set_xlabel('Age (years)'); ax.set_ylabel('Density'); ax.legend(frameon=False)
plt.show()

## A non-linear relationship with age?

Add age², then age² to age⁴. Watch the female coefficient and R².

In [ ]:
m3 = smf.ols('ln_wage ~ female + age + I(age**2)', data=cps_df).fit(cov_type='HC1')
m4 = smf.ols('ln_wage ~ female + age + I(age**2) + I(age**3) + I(age**4)', data=cps_df).fit(cov_type='HC1')
for name, m in [('(1)', m1), ('(2)', m2), ('(3)', m3), ('(4)', m4)]:
    print(name, 'female', round(m.params['female'], 3), '  R2', round(m.rsquared, 3))

## Does the gap have the same slope for both?

The age slope for women and men separately, then one pooled model with an interaction.

In [ ]:
women = smf.ols('ln_wage ~ age', data=cps_df[cps_df['female'] == 1]).fit(cov_type='HC1')
men = smf.ols('ln_wage ~ age', data=cps_df[cps_df['female'] == 0]).fit(cov_type='HC1')
inter = smf.ols('ln_wage ~ age * female', data=cps_df).fit(cov_type='HC1')
print('women', round(women.params['age'], 4), '  men', round(men.params['age'], 4))
print(inter.params.round(4))

## Interactions and non-linearities together

A 4th-order polynomial in age, interacted with gender. Then the predicted gap at different ages.

In [ ]:
full = smf.ols('ln_wage ~ female*age + female:I(age**2) + female:I(age**3) + female:I(age**4) '
               '+ age + I(age**2) + I(age**3) + I(age**4)', data=cps_df).fit(cov_type='HC1')

for a in [25, 30, 40, 50, 60]:
    p0 = full.predict(pd.DataFrame({'age': [a], 'female': [0]})).iloc[0]
    p1 = full.predict(pd.DataFrame({'age': [a], 'female': [1]})).iloc[0]
    print(a, f"gap {100 * (np.exp(p1 - p0) - 1):.0f}%")

## Predicting the gap by age, with confidence bands

In [ ]:
ages = np.linspace(25, 65, 50)

def predict_band(model, ages, female):
    X = pd.DataFrame({'age': ages, 'female': female})
    return model.get_prediction(X).summary_frame(alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, model, title in [(axes[0], inter, 'Linear interaction'), (axes[1], full, '4th-order polynomial interaction')]:
    for female, colour, label in [(0, '#2d6a4f', 'Male'), (1, '#457b9d', 'Female')]:
        band = predict_band(model, ages, female)
        ax.plot(ages, band['mean'], color=colour, label=label)
        ax.fill_between(ages, band['mean_ci_lower'], band['mean_ci_upper'], color=colour, alpha=0.15)
    ax.set_xlabel('Age (years)'); ax.set_title(title)
axes[0].set_ylabel('ln(earnings per hour)'); axes[0].legend(frameon=False)
plt.show()

# Part 2: presenting a regression table

The four gap models side by side, coefficients with stars and standard errors underneath.

In [ ]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

models = {'(1)': m1, '(2)': m2, '(3)': m3, '(4)': m4}
rows = {}
for name, label in [('female', 'Female'), ('age', 'Age')]:
    rows[label] = {k: f"{m.params[name]:.3f}{stars(m.pvalues[name])}" if name in m.params else ""
                   for k, m in models.items()}
    rows[label + ' SE'] = {k: f"({m.bse[name]:.3f})" if name in m.params else "" for k, m in models.items()}
rows['N'] = {k: f"{int(m.nobs):,}" for k, m in models.items()}
rows['R²'] = {k: f"{m.rsquared:.3f}" for k, m in models.items()}
print(pd.DataFrame(rows).T.to_markdown())

# Part 3: back to Vienna

The hotels, with distance and rating splines and star dummies, then the five hotels with the most negative residuals.

In [ ]:
hotels = pd.read_csv(base + "hotels_vienna.csv")
hotels = hotels[(hotels['accommodation_type'] == 'Hotel') & (hotels['city_actual'] == 'Vienna')
                & hotels['stars'].between(3, 4) & (hotels['price'] <= 600)].copy()
hotels['ln_price'] = np.log(hotels['price'])
hotels['dist_b1'] = np.maximum(hotels['distance'] - 1, 0)
hotels['dist_b4'] = np.maximum(hotels['distance'] - 4, 0)
hotels['rating_b35'] = np.maximum(hotels['rating'] - 3.5, 0)

model = smf.ols('ln_price ~ distance + dist_b1 + dist_b4 + rating + rating_b35 + C(stars)',
                data=hotels).fit(cov_type='HC1')
print(f"R2 = {model.rsquared:.2f}")

hotels['resid'] = model.resid
hotels.nsmallest(5, 'resid')[['hotel_id', 'price', 'resid', 'distance', 'stars', 'rating']]

## Comparing the hotel models

Build the model up one block at a time, and put the four side by side with the same table code as Part 2.

In [ ]:
h1 = smf.ols('ln_price ~ distance', data=hotels).fit(cov_type='HC1')
h2 = smf.ols('ln_price ~ distance + dist_b1 + dist_b4', data=hotels).fit(cov_type='HC1')
h3 = smf.ols('ln_price ~ distance + dist_b1 + dist_b4 + rating + rating_b35', data=hotels).fit(cov_type='HC1')
h4 = model

hmodels = {'(1)': h1, '(2)': h2, '(3)': h3, '(4)': h4}
rows = {}
for name in ['distance', 'dist_b1', 'dist_b4', 'rating', 'rating_b35', 'C(stars)[T.3.5]', 'C(stars)[T.4.0]']:
    rows[name] = {k: f"{m.params[name]:.3f}{stars(m.pvalues[name])}" if name in m.params else "" for k, m in hmodels.items()}
rows['N'] = {k: f"{int(m.nobs)}" for k, m in hmodels.items()}
rows['R²'] = {k: f"{m.rsquared:.3f}" for k, m in hmodels.items()}
print(pd.DataFrame(rows).T.to_markdown())

# Do the stars matter jointly?
print(h4.f_test("C(stars)[T.3.5] = 0, C(stars)[T.4.0] = 0"))

# How many of the straight line's five best deals survive in the full model?
best1 = set(hotels.assign(r=h1.resid).nsmallest(5, 'r')['hotel_id'])
best4 = set(hotels.nsmallest(5, 'resid')['hotel_id'])
print(len(best1 & best4), "of 5")